<a href="https://colab.research.google.com/github/MHamza-Ahmad/Flyrank-Internship/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MHamza-Ahmad/Flyrank-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [10]:
import os
# For testing purposes, setting a placeholder HF_TOKEN.
# You should replace this with your actual Hugging Face token set in Colab secrets for real data access.
if 'HF_TOKEN' not in os.environ:
    os.environ['HF_TOKEN'] = 'hf_THISISAPLACEHOLDERDONTUSEITINPROD'


In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np

print("Generating robust Search Console dataset for code verification...")
np.random.seed(42) # Adding seed for reproducible randomness
dates = pd.date_range(start='2026-03-01', end='2026-03-31')

# We add random noise to impressions and clicks so the data behaves like real-world data
base_impressions = np.array([100, 50, 200] * len(dates))
impressions_with_noise = base_impressions + np.random.randint(-15, 15, size=len(base_impressions))

base_clicks = np.array([10, 0, 45] * len(dates))
clicks_with_noise = base_clicks + np.random.randint(-3, 4, size=len(base_clicks))

df = pd.DataFrame({
    'date': dates.repeat(3),
    'query': ['seo tool', 'forecasting', 'machine learning'] * len(dates),
    'url': ['/tools', '/blog/forecast', '/blog/ml'] * len(dates),
    'impressions': np.clip(impressions_with_noise, 1, None), # Ensure no negative impressions
    'position': [3.5, 12.0, 1.2] * len(dates),
    'clicks': np.clip(clicks_with_noise, 0, None), # Ensure no negative clicks
    'available': [True, False, True] * len(dates)
})

print(f"Dataset loaded successfully with {len(df)} rows.")
print(f"Date Window: {df['date'].min()} to {df['date'].max()}")

Generating robust Search Console dataset for code verification...
Dataset loaded successfully with 93 rows.
Date Window: 2026-03-01 00:00:00 to 2026-03-31 00:00:00


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

features = ['impressions', 'position', 'available']
label = 'clicks'
context = ['date', 'query', 'url']
excluded = ['transaction_id']

# Verify the retained columns exist in the dataframe
expected_columns = features + [label] + context
missing_cols = [col for col in expected_columns if col not in df.columns]

assert len(missing_cols) == 0, f"Missing columns in dataset: {missing_cols}"
print("All contract fields are present in the dataset.")

All contract fields are present in the dataset.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# --- PART 1: The Three Verification Queries ---

# Query 1: The Grain (One row really is Date + Query + URL)
duplicate_grain_count = df.duplicated(subset=['date', 'query', 'url']).sum()
print(f"1. Grain Check (Duplicates on Date/Query/URL): {duplicate_grain_count}")

# Query 2: Row Count and Date Span
print(f"2. Data Span: {len(df):,} rows from {df['date'].min()} to {df['date'].max()}")

# Query 3: Availability (Filter with IS TRUE)
surviving_rows = len(df[df['available'] == True])
print(f"3. Availability Check: {surviving_rows:,} rows survive the 'IS TRUE' filter.")
print("-" * 40)

# --- PART 2: Feature Engineering & The Leakage Trap ---

df_features = df.copy()
df_features['query_length'] = df_features['query'].astype(str).apply(len)
df_features['day_of_week'] = pd.to_datetime(df_features['date']).dt.dayofweek
print("Features built successfully.")

# THE TRAP: Adding a label-derived column on purpose
# This literally copies the target we are trying to predict
df_features['TRAP_derived_clicks'] = df_features['clicks'] * 1.0

from sklearn.linear_model import LinearRegression
model = LinearRegression()
X_trap = df_features[['impressions', 'position', 'TRAP_derived_clicks']]
y = df_features['clicks']

model.fit(X_trap, y)
trap_score = model.score(X_trap, y)
print(f"Trap Score (R^2) with leaked feature: {trap_score:.4f} (Dangerously close to 1.0)")

# Springing the trap: Delete the leaked column and keep the honest features
# (We explicitly do NOT use CTR here, because calculating CTR requires knowing the clicks beforehand)
df_features = df_features.drop(columns=['TRAP_derived_clicks'])
X_honest = df_features[['impressions', 'position', 'query_length', 'day_of_week']]

model.fit(X_honest, y)
honest_score = model.score(X_honest, y)
print(f"Honest Score (R^2) after removing leakage: {honest_score:.4f} (A realistic ML score)")

1. Grain Check (Duplicates on Date/Query/URL): 0
2. Data Span: 93 rows from 2026-03-01 00:00:00 to 2026-03-31 00:00:00
3. Availability Check: 62 rows survive the 'IS TRUE' filter.
----------------------------------------
Features built successfully.
Trap Score (R^2) with leaked feature: 1.0000 (Dangerously close to 1.0)
Honest Score (R^2) after removing leakage: 0.9914 (A realistic ML score)


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

total_impressions = df['impressions'].sum()
print(f"Limitation Acknowledged: The {total_impressions:,} impressions observed exclude anonymized Google queries.")

Limitation Acknowledged: The 10,912 impressions observed exclude anonymized Google queries.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.